In [5]:
import pandas as pd
import numpy as np 
import warnings
warnings.filterwarnings('ignore')

#reload the raw dataset freash
df = pd.read_excel(
    '../data/raw/NFHS_5_India_Districts_Factsheet_Data.xls',
    engine='xlrd',
    header=0
)

print("Data loaded fresh!")
print(f"shape: {df.shape}")

Data loaded fresh!
shape: (707, 109)


In [7]:
#find how many * values exist in each column
# * means government suppressed that data (small sample size)

star_count = {}

for col in df.columns:
    count = (df[col] == '*').sum()
if count >0:
     star_count[col] = count

print(f"columns containing * values: {len(star_count)}")
print(f"\ncolumn name : * count")
print("_" * 50)
for col, count in star_count.items():
    print(f"{col[:50]} : {count}")


columns containing * values: 0

column name : * count
__________________________________________________


investigate

investigate 

In [9]:
#find all object type columns (should be float but aren't )
obj_cols = df.select_dtypes(include='object').columns.tolist()
print(f"object columns count: {len(obj_cols)}")
print()

#look at unique values in first 3 object columns
#to see what's actually stored there
for col in obj_cols[:3]:
      print(f"column: {col[:60]}")
      print(f"unique values: {df[col].unique()[:10]}")
      print()


object columns count: 31

column: District Names
unique values: <ArrowStringArray>
[               'Nicobars', 'North & Middle Andaman ',
          'South Andaman ',             'Srikakulam ',
           'Vizianagaram ',          'Visakhapatnam ',
          'East Godavari ',          'West Godavari ',
                'Krishna ',                 'Guntur ']
Length: 10, dtype: str

column: State/UT
unique values: <ArrowStringArray>
[             'Andaman & Nicobar Islands',
                         'Andhra Pradesh',
                      'Arunachal Pradesh',
                                  'Assam',
                                  'Bihar',
                             'Chandigarh',
                           'Chhattisgarh',
 'Dadra and Nagar Haveli & Daman and Diu',
                                    'Goa',
                                'Gujarat']
Length: 10, dtype: str

column: Deaths in the last 3 years registered with the civil authori
unique values: [83.22 -92.55 92.21 71.03 81.

find and replace all* values

In [ ]:
#replace * with NaN (proper missing value)
#Nan = not a number = pandas standard for missing data
#NaN means not a number

#first check what non-numeric values exist in object columns
problem_cols = obj_cols[2:]  #skip district names and state/ut
print(f"problem columns to fix: {len(problem_cols)}")

#replace * with NaN across entire dataframe

df_clean = df.copy()  #(always work on a copy, never the originals)
df_clean =df_clean.replace('*', np.nan)

print("* values replaced with NaN successfully")

#verity check if * still exists
remaining = (df_clean == '*').sum().sum()
print(f"remaining * values: {remaining}")

problem columns to fix: 29
* values replaced with NaN successfully
remaining * values: 0


convert object columns to float

In [11]:
#convert all 29 problem columns from text to numbers
for col in problem_cols:
     df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

     #verify conversion worked
     obj_remaining = df_clean.select_dtypes(include='object').columns.tolist()
     print(f"object columns remaining: {len(obj_remaining)}")
     print(f"These should only be: {obj_remaining}")
     

object columns remaining: 30
These should only be: ['District Names', 'State/UT', 'Children age 5 years who attended pre-primary school during the school year 2019-20 (%)', 'Births in the 5 years preceding the survey that are third or higher order (%)', 'Current users ever told about side effects of current method of family planning8 (%)', 'Average out-of-pocket expenditure per delivery in a public health facility (for last birth in the 5 years before the survey) (Rs.)', 'Children born at home who were taken to a health facility for a check-up within 24 hours of birth (for last birth in the 5 years before the survey} (%)', 'Births in a private health facility that were delivered by caesarean section (in the 5 years before the survey) (%)', "Children age 12-23 months fully vaccinated based on information from either vaccination card or mother's recall11 (%)", 'Children age 12-23 months fully vaccinated based on information from vaccination card only12 (%)', 'Children age 12-23 months wh

rename columns to short clean names

In [13]:
#create short clean column names
#current names are 100+ characters long -impossible to code with
#new names are descriptive but short

column_mapping = {
    'District Names': 'district',
    'State/UT': 'state',
    'Number of Households surveyed': 'households_surveyed',
    'Number of Women age 15-49 years interviewed': 'women_interviewed',
    'Number of Men age 15-54 years interviewed': 'men_interviewed',
    'Female population age 6 years and above who ever attended school (%)': 'female_school_attendance',
    'Population below age 15 years (%)': 'pop_below_15',
    'Sex ratio of the total population (females per 1,000 males)': 'sex_ratio',
    'Children under 5 years who are stunted (height-for-age)18 (%)': 'stunting_pct',
    'Children under 5 years who are wasted (weight-for-height)18 (%)': 'wasting_pct',
    'Children under 5 years who are underweight (weight-for-age)18 (%)': 'underweight_pct',
    'Children age 6-59 months who are anaemic (<11.0 g/dl)22 (%)': 'child_anaemia_pct',
    'Non-pregnant women age 15-49 years who are anaemic (<12.0 g/dl)22 (%)': 'women_anaemia_pct',
    'All women age 15-49 years who are anaemic22 (%)': 'all_women_anaemia_pct',
    'Population living in households with electricity (%)': 'electricity_pct',
    'Population living in households with an improved drinking-water source1 (%)': 'clean_water_pct',
    'Population living in households that use an improved sanitation facility2 (%)': 'sanitation_pct',
    'Households using clean fuel for cooking3 (%)': 'clean_fuel_pct',
    'Women (age 15-49) who are literate4 (%)': 'women_literacy_pct',
    'Women age 20-24 years married before age 18 years (%)': 'child_marriage_pct',
    'Institutional births (in the 5 years before the survey) (%)': 'institutional_births_pct',
    'Mothers who had at least 4 antenatal care visits  (for last birth in the 5 years before the survey) (%)': 'antenatal_4visits_pct',
    'Women age 15-19 years who were already mothers or pregnant at the time of the survey (%)': 'teen_pregnancy_pct',
    'Households with any usual member covered under a health insurance/financing scheme (%)': 'health_insurance_pct',
    'Women age 15 years and above who use any kind of tobacco (%)': 'women_tobacco_pct',
    'Men age 15 years and above who use any kind of tobacco (%)': 'men_tobacco_pct',
    'Women age 15 years and above who consume alcohol (%)': 'women_alcohol_pct',
    'Men age 15 years and above who consume alcohol (%)': 'men_alcohol_pct',
}

# Apply the renaming
df_clean = df_clean.rename(columns=column_mapping)

print("Columns renamed successfully!")
print(f"\nSample new column names:")
print(list(df_clean.columns[:10]))

Columns renamed successfully!

Sample new column names:
['district', 'state', 'households_surveyed', 'women_interviewed', 'men_interviewed', 'female_school_attendance', 'pop_below_15', ' Sex ratio of the total population (females per 1,000 males)', 'Sex ratio at birth for children born in the last five years (females per 1,000 males)', 'Children under age 5 years whose birth was registered with the civil authority (%)']


handle missing values (NaN)

In [27]:
# Check missing values after cleaning
missing_after = df_clean.isnull().sum()
missing_pct = (missing_after / len(df_clean) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing_after,
    'missing_pct': missing_pct
})

# Show only columns with missing values
has_missing = missing_report[missing_report['missing_count'] > 0]
print(f"Columns with missing values: {len(has_missing)}")
print()
print(has_missing.sort_values('missing_pct', ascending=False).head(20))

Columns with missing values: 0

Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


Rules:

Missing > 30% → DROP the column
Missing < 30% → FILL with median

apply the rules

In [28]:
# Fill remaining missing values with median
cols_to_fill = has_missing[has_missing['missing_pct'] <= threshold].index.tolist()
print(f"Columns to FILL with median: {len(cols_to_fill)}")

for col in cols_to_fill:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

# Final check
print(f"\nShape after cleaning: {df_clean.shape}")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")

Columns to FILL with median: 0

Shape after cleaning: (707, 101)
Remaining missing values: 0


save the cleaned file

In [29]:
#save  cleaned data to processed folder
#this is your production-ready dataset

output_path='../data/processed/nfhs5_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"Cleaned data saved!")
print(f"Location: {output_path}")
print(f"Final shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns[:10])}...")

Cleaned data saved!
Location: ../data/processed/nfhs5_cleaned.csv
Final shape: (707, 101)
Columns: ['district', 'state', 'households_surveyed', 'women_interviewed', 'men_interviewed', 'female_school_attendance', 'pop_below_15', ' Sex ratio of the total population (females per 1,000 males)', 'Sex ratio at birth for children born in the last five years (females per 1,000 males)', 'Children under age 5 years whose birth was registered with the civil authority (%)']...
